# Visualisation de données en Python: comparaison d'outils

Notebook de l'article [Visualisation de données en Python: comparaison d'outils](https://moncoachdata.com/blog/visualisation-de-donnees-avec-python/) (MonCoachData). Exécute les cellules dans l'ordre.

Le code est testé avec Python 3.12, Pandas 3, Matplotlib 3.10, Seaborn 0.13, Bokeh 3, Cartopy 0.26 et NetworkX 3. Il fonctionne aussi sur Google Colab (la première cellule installe ce qui manque).

Python est un langage scientifique très mature. On y trouve des bibliothèques pour tout un tas d'usages différents, notamment le Machine Learning et l'analyse de data.

La **visualisation de données Python** est une partie importante du travail : elle sert à explorer les données et à communiquer les résultats.

Au cours des dernières années, de nombreuses bibliothèques de visualisation de données ont vu le jour en Python pour combler son retard sur le langage R. [Matplotlib](https://matplotlib.org/) est devenue la principale (voir aussi notre article sur la [visualisation de données avec Matplotlib](https://moncoachdata.com/blog/matplotlib-visualisation-de-donnees/)).

Mais il en existe d'autres, qui répondent à des besoins plus précis :

- [vispy](https://vispy.org/) : visualisation scientifique accélérée par GPU ;
- [Bokeh](https://bokeh.org/) : graphiques interactifs dans le navigateur ;
- [Seaborn](https://seaborn.pydata.org/) : graphiques statistiques, construit sur Matplotlib ;
- [pygal](https://github.com/Kozea/pygal) : graphiques SVG soignés ;
- [folium](https://github.com/python-visualization/folium) : cartes interactives ;
- [networkx](https://github.com/networkx/networkx) : graphes et réseaux.

Dans cet article, nous utiliserons chacune de ces bibliothèques pour créer des graphiques, et nous découvrirons leurs forces et leurs faiblesses.

Quelle est la plus performante ? Comment tirer le meilleur de Python en termes de visualisation de données ?

**En bref :** en Python, Matplotlib est la bibliothèque de visualisation de référence : très flexible, mais il faut plus de commandes pour obtenir des graphiques attrayants. Seaborn est plus agréable par défaut et adapté aux graphiques statistiques, Bokeh et folium rendent graphiques et cartes interactifs, pygal soigne la présentation (SVG) et networkx dessine des réseaux. L'article les compare sur les données OpenFlights (routes, aéroports, compagnies aériennes).

## Quelles données utilise-t-on pour comparer les bibliothèques ?

Avant de rentrer dans le vif du sujet, jetons un coup d'œil au dataset avec lequel nous allons travailler. Nous utiliserons les données d'[OpenFlights](https://openflights.org/) (routes aériennes, aéroports et compagnies aériennes).

Les trois fichiers viennent du dépôt GitHub d'OpenFlights (`airports.dat`, `airlines.dat` et `routes.dat`). Le notebook les télécharge directement, sans compte ni fichier à préparer.

## Installer les bibliothèques manquantes

Pandas, Matplotlib, Seaborn, NumPy et SciPy sont déjà dans Colab. Les autres s'installent en une ligne (inutile si tu les as déjà) :

In [ ]:
%pip install -q bokeh pygal folium cartopy networkx

Chaque ligne du fichier des routes correspond à une route aérienne entre deux aéroports. Chaque ligne du fichier des aéroports correspond à un aéroport dans le monde, avec ses informations. Chaque ligne du fichier des compagnies représente tout simplement une compagnie aérienne.

### Comment lire les données ?

Les données n'ont pas d'en-têtes de colonne. Nous les ajoutons donc manuellement en les affectant à l'attribut `columns`.

Nous voulons lire chaque colonne en tant que chaîne de caractères : cela facilitera la comparaison des données plus tard, quand nous devrons faire correspondre les lignes à partir de leur identifiant (`id`). Pour cela, nous définissons le paramètre `dtype` lors de la lecture des données.

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/"

# Les fichiers n'ont pas d'en-têtes : on les ajoute en affectant l'attribut columns.
# dtype=str : chaque colonne est lue comme du texte, ce qui facilite la comparaison des identifiants.
airports = pd.read_csv(BASE + "airports.dat", header=None, dtype=str)
airports.columns = ["id", "name", "city", "country", "code", "icao", "latitude", "longitude",
                    "altitude", "offset", "dst", "timezone", "type", "source"]

airlines = pd.read_csv(BASE + "airlines.dat", header=None, dtype=str)
airlines.columns = ["id", "name", "alias", "iata", "icao", "callsign", "country", "active"]

routes = pd.read_csv(BASE + "routes.dat", header=None, dtype=str)
routes.columns = ["airline", "airline_id", "source", "source_id", "dest", "dest_id",
                  "codeshare", "stops", "equipment"]

print(airports.shape, airlines.shape, routes.shape)

Regardons le résultat pour chaque DataFrame :

In [ ]:
airports.head()

In [ ]:
airlines.head()

In [ ]:
routes.head()

Nous pouvons faire de nombreuses explorations intéressantes avec chaque dataset, mais ce sera plus bénéfique en les combinant. La bibliothèque Pandas nous aide dans cette analyse, car elle permet de filtrer facilement les tableaux et d'appliquer des fonctions sur des colonnes.

Nous allons nous intéresser à quelques métriques, comme les compagnies aériennes et les itinéraires aériens.

Avant cela, nous devons faire un peu de nettoyage de données (*data cleaning*) :

Cette ligne nous assure d'avoir uniquement des données numériques dans la colonne `airline_id`.

In [ ]:
# On ne garde que les routes dont l'identifiant de compagnie est numérique (OpenFlights note \N les valeurs inconnues)
routes = routes[routes["airline_id"] != "\\N"]
routes.shape

## Comment réaliser un histogramme avec Matplotlib ?

Nous pouvons maintenant aller plus loin et faire des graphiques pour explorer nos données. Pour le premier, nous utiliserons Matplotlib.

Matplotlib est une bibliothèque assez basique pour tracer des graphiques avec Python. Il faut donc généralement plus de commandes pour faire des graphiques attrayants qu'avec d'autres bibliothèques.

D'un autre côté, tu peux faire presque n'importe quel type de graphique avec Matplotlib : cette flexibilité se paie en nombre de commandes nécessaires pour tracer un graphique un peu plus élaboré.

Nous allons d'abord faire un histogramme montrant la distribution des longueurs des routes aériennes. Pour rappel, un histogramme divise toutes les longueurs d'itinéraire en intervalles (ou « bins ») et compte le nombre d'itinéraires qui tombent dans chaque intervalle.

Cela nous indique si les compagnies aériennes effectuent plutôt des trajets courts ou plutôt des trajets longs.

Pour cela, nous devons d'abord calculer les longueurs des routes aériennes. La première étape est une formule de distance : nous utiliserons la formule d'haversine, qui calcule la distance entre deux points d'une sphère à partir de leurs longitudes et latitudes.

In [ ]:
import math

def haversine(lon1, lat1, lon2, lat2):
    # Distance (en km) entre deux points d'une sphère, à partir de leurs longitudes et latitudes
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    km = 6367 * c
    return km

Ensuite, nous écrivons une fonction qui calcule la distance entre les aéroports `source` et `dest` pour un seul itinéraire.

Pour cela, nous récupérons les `source_id` et `dest_id` des aéroports dans le DataFrame `routes`. Puis nous les faisons correspondre à la colonne `id` du DataFrame `airports` pour obtenir la latitude et la longitude de ces aéroports. Il ne reste plus qu'à appliquer la formule.

Voici la fonction :

La fonction peut échouer s'il y a une valeur invalide dans les colonnes `source_id` ou `dest_id`. D'où le bloc `try` / `except` pour éviter l'erreur.

In [ ]:
def calc_dist(row):
    dist = 0
    try:
        # On récupère les aéroports de départ et d'arrivée à partir de leur identifiant
        source = airports[airports["id"] == row["source_id"]].iloc[0]
        dest = airports[airports["id"] == row["dest_id"]].iloc[0]
        dist = haversine(float(dest["longitude"]), float(dest["latitude"]),
                         float(source["longitude"]), float(source["latitude"]))
    except (ValueError, IndexError):
        pass
    return dist

Pour finir, nous utilisons la méthode `apply` de Pandas pour appliquer la fonction de calcul de la distance (`calc_dist`) à l'ensemble des routes. Cela nous donne un objet `Series` contenant toutes les longueurs de route, exprimées en kilomètres.

In [ ]:
route_lengths = routes.apply(calc_dist, axis=1)
route_lengths.head()

Maintenant que nous avons une série contenant la longueur des itinéraires, nous pouvons créer un histogramme. Il classe les valeurs dans des intervalles et compte le nombre d'itinéraires dans chacun.

Regardons le résultat :

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.hist(route_lengths, bins=20)
plt.xlabel("Longueur de la route (km)")
plt.ylabel("Nombre de routes")
plt.show()

Nous importons `pyplot` depuis Matplotlib, que nous renommons `plt`. Nous configurons ensuite Matplotlib pour afficher les tracés dans le notebook Jupyter avec `%matplotlib inline`.

Finalement, nous traçons l'histogramme avec `plt.hist(route_lengths, bins = 20)`. Comme tu peux le voir, les compagnies aériennes effectuent beaucoup plus de trajets courts que de trajets longs.

## Comment faire un histogramme plus soigné avec Seaborn ?

Nous pouvons réaliser un graphique similaire avec Seaborn, une bibliothèque plus élaborée qui s'appuie sur Matplotlib et simplifie certains graphiques statistiques.

Nous utilisons la fonction `histplot` avec `kde=True` pour tracer un histogramme avec une estimation de densité de noyau (KDE) : une courbe qui est une version lissée de l'histogramme, plus facile à analyser. (Dans les anciennes versions de Seaborn, on utilisait `distplot`, déprécié depuis Seaborn 0.11.)

In [ ]:
import seaborn as sns

sns.histplot(route_lengths, bins=20, kde=True)
plt.xlabel("Longueur de la route (km)")
plt.ylabel("Nombre de routes")
plt.show()

Comme tu peux le voir, Seaborn est par défaut esthétiquement plus agréable que Matplotlib : il permet d'obtenir rapidement de plus beaux graphiques. C'est aussi une bonne bibliothèque si tu as besoin d'approfondir ton travail avec un angle statistique.

## Comment faire un diagramme en barres, puis le rendre interactif avec Bokeh ?

Les histogrammes, c'est bien… mais voir la longueur moyenne d'un itinéraire par compagnie aérienne, c'est mieux 🙂

Pour cela, nous utilisons un diagramme en barres, avec une barre par compagnie. Cela nous permet de voir quelles compagnies sont nationales et lesquelles sont internationales.

Nous utilisons d'abord Pandas pour déterminer la longueur moyenne d'un itinéraire par compagnie aérienne.

Nous créons d'abord un nouveau DataFrame avec les longueurs d'itinéraire et les identifiants des compagnies aériennes. Nous divisons `route_length_df` en groupes à partir de `airline_id`, ce qui crée un DataFrame par compagnie.

In [ ]:
import numpy as np

route_length_df = pd.DataFrame({"length": route_lengths, "airline_id": routes["airline_id"]})

Nous utilisons ensuite la fonction `agg` de Pandas pour calculer la moyenne de la colonne `length` de chacun de ces DataFrames, puis nous ajoutons chaque résultat dans un nouveau DataFrame, `airline_route_lengths`. Nous trions enfin les données pour que les itinéraires les plus longs sortent en premier.

In [ ]:
airline_route_lengths = route_length_df.groupby("airline_id").agg({"length": "mean"})
airline_route_lengths = airline_route_lengths.sort_values("length", ascending=False)
airline_route_lengths.head()

Nous pouvons enfin tracer notre diagramme en barres avec Matplotlib :

In [ ]:
plt.bar(range(airline_route_lengths.shape[0]), airline_route_lengths["length"])
plt.xlabel("Compagnies aériennes (triées par longueur moyenne)")
plt.ylabel("Longueur moyenne des routes (km)")
plt.show()

La méthode `plt.bar` de Matplotlib trace la longueur moyenne d'itinéraire de chaque compagnie aérienne (`airline_route_lengths["length"]`).

Le problème de ce graphique, c'est que nous ne pouvons pas déterminer facilement la longueur moyenne de vol d'une compagnie donnée. Il faudrait ajouter des intitulés sur l'axe horizontal, ce qui n'est pas évident au vu du nombre de compagnies.

Une solution est de rendre le graphique interactif, pour pouvoir zoomer et voir les noms des compagnies. Nous pouvons utiliser la bibliothèque Bokeh, qui rend la création de graphiques interactifs et zoomables assez facile.

Pour utiliser Bokeh, nous devons d'abord préparer nos données :

In [ ]:
def lookup_name(airline_id):
    # Nom de la compagnie à partir de son identifiant
    try:
        return airlines.loc[airlines["id"] == airline_id, "name"].iloc[0]
    except IndexError:
        return "Inconnue"

airline_route_lengths["name"] = airline_route_lengths.index.map(lookup_name)
airline_route_lengths["id"] = airline_route_lengths.index
airline_route_lengths = airline_route_lengths.reset_index(drop=True)
airline_route_lengths.head()

Le code ci-dessus récupère le nom de chaque ligne de `airline_route_lengths` et l'ajoute dans une colonne `name`. Nous ajoutons aussi une colonne `id` pour pouvoir faire cette recherche (la fonction `apply` ne s'applique qu'aux identifiants de l'index).

Enfin, nous réinitialisons l'index pour avoir des valeurs uniques : Bokeh ne fonctionne pas correctement sans cela.

Nous pouvons maintenant passer à la partie graphique :

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool

output_notebook()

source = ColumnDataSource(airline_route_lengths.assign(rang=airline_route_lengths.index))
p = figure(title="Longueur moyenne des routes par compagnie aérienne", width=800, height=450,
           x_axis_label="Compagnies (triées par longueur moyenne)", y_axis_label="Longueur moyenne (km)",
           tools="pan,wheel_zoom,box_zoom,reset")
p.vbar(x="rang", top="length", width=0.9, source=source)
p.add_tools(HoverTool(tooltips=[("Compagnie", "@name"), ("Longueur moyenne (km)", "@length{0}")]))
show(p)

Nous appelons `output_notebook` pour configurer Bokeh afin qu'il affiche le graphique interactif dans le notebook. Ensuite, avec `bokeh.plotting`, nous faisons un diagramme en barres à partir de certaines colonnes de notre DataFrame, avec une infobulle qui donne le nom de la compagnie au survol. Enfin, la fonction `show` affiche le graphique.

Le graphique généré n'est pas une image : c'est un widget JavaScript. C'est pourquoi la capture d'écran ci-dessous, contrairement au graphique réel, n'est pas interactive.

Avec ce graphique, nous pouvons zoomer et voir quelles compagnies aériennes effectuent les plus longs trajets. Sur l'image ci-dessus, les noms semblent se superposer, mais ils sont beaucoup plus faciles à lire quand tu zoomes.

## Comment faire un graphique à barres horizontales avec Pygal ?

Pygal est une bibliothèque Python de visualisation qui rend nos graphiques rapidement attrayants. Nous pouvons l'utiliser pour faire une ventilation des itinéraires par longueur.

Nous séparons d'abord nos itinéraires en trois catégories : court (moins de 2000 km), moyen (de 2000 à 10000 km) et long (plus de 10000 km). Puis nous calculons la part de chacune.

In [ ]:
# Catégories : court (moins de 2000 km), moyen (2000 à 10000 km) et long (plus de 10000 km)
long_routes = len([k for k in route_lengths if k > 10000]) / len(route_lengths)
medium_routes = len([k for k in route_lengths if k < 10000 and k > 2000]) / len(route_lengths)
short_routes = len([k for k in route_lengths if k < 2000]) / len(route_lengths)

print(f"Long : {long_routes:.1%} | Moyen : {medium_routes:.1%} | Court : {short_routes:.1%}")

Nous pouvons ensuite tracer chacune de ces catégories dans un graphique à barres horizontales Pygal :

In [ ]:
import pygal
from IPython.display import SVG

chart = pygal.HorizontalBar()
chart.title = "Itinéraires aériens : courts, moyens et longs"
chart.add("Long", long_routes * 100)
chart.add("Moyen", medium_routes * 100)
chart.add("Court", short_routes * 100)
chart.render_to_file("routes.svg")
SVG(filename="routes.svg")

Ci-dessus, nous créons d'abord un graphique vide, puis nous ajoutons des éléments : un titre et des barres. Chaque barre a un pourcentage (sur 100) indiquant la fréquence de ce type de route.

Enfin, nous enregistrons le graphique dans un fichier et utilisons les capacités d'affichage SVG de Jupyter pour le charger et l'afficher.

Ce graphique est plus joli que les graphiques Matplotlib par défaut, mais nous avons dû écrire plus de code pour le créer. Pygal est très bon pour des graphiques nécessitant une qualité de présentation supérieure.

## Comment faire un nuage de points avec Matplotlib et Seaborn ?

Les nuages de points permettent de comparer des colonnes de données. Nous pouvons par exemple faire un nuage de points simple pour comparer l'identifiant (`id`) de la compagnie aérienne à la longueur du nom de la compagnie :

Nous calculons d'abord la longueur de chaque nom avec la méthode `apply` de Pandas, qui retourne le nombre de caractères de chaque nom de compagnie.

In [ ]:
name_lengths = airlines["name"].apply(lambda x: len(str(x)))

Nous réalisons ensuite un nuage de points (avec Matplotlib) comparant les identifiants des compagnies à la longueur de leur nom. Lors du tracé, nous convertissons la colonne `id` de `airlines` en entier, car le graphique a besoin de valeurs numériques sur l'axe des x.

In [ ]:
plt.scatter(airlines["id"].astype(int), name_lengths)
plt.xlabel("Identifiant de la compagnie")
plt.ylabel("Longueur du nom")
plt.show()

Nous voyons qu'un certain nombre de noms plus longs apparaissent parmi les premiers identifiants. Cela pourrait signifier que les compagnies fondées il y a plus longtemps ont tendance à avoir des noms plus longs.

Nous pouvons vérifier cette intuition avec Seaborn. Il propose une version améliorée du nuage de points, le `jointplot`, qui montre à quel point deux variables sont corrélées, ainsi que la distribution de chacune.

In [ ]:
data = pd.DataFrame({"lengths": name_lengths, "ids": airlines["id"].astype(int)})
sns.jointplot(x="ids", y="lengths", data=data)
plt.show()

# Coefficient de corrélation de Pearson (r) et son carré (r²)
from scipy.stats import pearsonr
r, p_value = pearsonr(data["ids"], data["lengths"])
print(f"r = {r:.3f}, r² = {r**2:.3f}")

Le graphique ci-dessus montre qu'il n'y a pas de corrélation réelle entre les deux variables : la valeur de r² est faible.

## Comment dessiner une carte des aéroports avec Cartopy et folium ?

Nos données se prêtent naturellement à la cartographie : nous avons des paires de latitude et de longitude pour les aéroports, ainsi que pour les points de départ et de destination.

La première carte montre les aéroports du monde entier. Nous la réalisons avec [Cartopy](https://scitools.org.uk/cartopy/docs/latest/), qui s'appuie sur Matplotlib et permet de dessiner des cartes du monde et d'y ajouter des points. (L'article d'origine utilisait Basemap, qui n'est plus maintenu.)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

airports["longitude"] = airports["longitude"].astype(float)
airports["latitude"] = airports["latitude"].astype(float)

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Mercator(min_latitude=-80, max_latitude=80))
ax.set_extent([-180, 180, -58, 80], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#eeeeee")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

ax.scatter(airports["longitude"], airports["latitude"], s=2, color="red", transform=ccrs.PlateCarree())
plt.show()

Dans le code ci-dessus, nous dessinons d'abord une carte du monde avec une [projection de Mercator](https://fr.wikipedia.org/wiki/Projection_de_Mercator), une façon de projeter toute la carte du monde sur une surface 2D. Puis nous dessinons les aéroports sur la carte avec des points rouges.

Le problème de cette carte est qu'il est difficile de voir où se trouve chaque aéroport : dans les zones à forte densité, les points se fondent en une masse rouge.

Comme avec Bokeh, il existe une bibliothèque de cartographie interactive, `folium`, que nous pouvons utiliser pour zoomer sur la carte et trouver des aéroports individuels.

Folium utilise `leaflet.js` pour créer une carte entièrement interactive. Tu peux cliquer sur chaque aéroport pour voir son nom dans une fenêtre contextuelle.

In [ ]:
import folium

# Obtenir une carte du monde basique
airports_map = folium.Map(location=[30, 0], zoom_start=2)

# Dessiner des marqueurs sur la carte
for name, row in airports.iterrows():
    # problème avec cet aéroport sur la carte, on le supprime
    if row["name"] != "South Pole Station":
        folium.CircleMarker(location=[float(row["latitude"]), float(row["longitude"])],
                            popup=row["name"]).add_to(airports_map)

# Créer et afficher la carte
airports_map.save("airports.html")
airports_map

Une capture d'écran est montrée ci-dessus, mais la carte réelle est beaucoup plus impressionnante. Folium permet aussi de modifier assez largement les options pour créer de meilleurs marqueurs ou ajouter d'autres éléments à la carte.

## Comment dessiner les routes aériennes avec de grands cercles ?

Ce qui serait vraiment cool, c'est de voir toutes les routes aériennes sur une carte. Heureusement, nous pouvons utiliser Cartopy pour cela.

Nous dessinons de grands cercles ([orthodromies](https://fr.wikipedia.org/wiki/Orthodromie)) reliant les aéroports de départ et de destination : chaque cercle montre l'itinéraire d'un avion de ligne. Il y a malheureusement tellement de routes que toutes les montrer ferait une sacrée pagaille. Nous allons donc afficher les 3000 premières routes.

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Mercator(min_latitude=-80, max_latitude=80))
ax.set_extent([-180, 180, -58, 80], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#eeeeee")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

airports_idx = airports.set_index("id")

# On ne dessine que les 3000 premières routes
for _, row in routes.head(3000).iterrows():
    try:
        source = airports_idx.loc[row["source_id"]]
        dest = airports_idx.loc[row["dest_id"]]
    except KeyError:
        continue
    # Filtre : on évite les routes trop longues, qui masqueraient les autres
    if abs(source["longitude"] - dest["longitude"]) < 90:
        ax.plot([source["longitude"], dest["longitude"]], [source["latitude"], dest["latitude"]],
                color="b", linewidth=0.3, transform=ccrs.Geodetic())
plt.show()

Le code ci-dessus dessine une carte, puis les routes par-dessus. Nous ajoutons des filtres pour éviter que les routes trop longues n'obscurcissent les autres.

## Comment dessiner un diagramme de réseau avec networkx ?

La dernière exploration est un diagramme de réseau des aéroports. Chaque aéroport est un nœud du réseau, et nous traçons une liaison entre deux nœuds s'il existe une route entre les aéroports.

S'il y a plusieurs routes, nous ajoutons un poids à la liaison pour montrer que les aéroports sont plus connectés. Nous utilisons la bibliothèque `networkx` pour cela.

Tout d'abord, nous devons calculer les poids entre les aéroports :

In [ ]:
# Initialisation du dictionnaire weights contenant les poids
weights = {}
# On garde une trace des clés qui ont été ajoutées une fois.
# Nous voulons seulement des liaisons avec un poids de plus de 1 pour garder une taille de réseau gérable.
added_keys = set()

# Itérer sur chaque itinéraire
for name, row in routes.iterrows():
    # Extraire les ids de source et dest des aéroports
    source = row["source_id"]
    dest = row["dest_id"]

    # Créer une clé pour le dictionnaire weights : elle correspond à une liaison (départ_arrivée)
    key = "{0}_{1}".format(source, dest)
    # Si la clé est déjà présente dans weights, incrémenter le poids
    if key in weights:
        weights[key] += 1
    # Si la clé est présente dans added_keys, initialiser la clé dans weights avec un poids de 2
    elif key in added_keys:
        weights[key] = 2
    # Si la clé n'est pas encore dans added_keys, on l'ajoute.
    # Cela nous assure de ne pas ajouter de liaisons avec un poids de 1
    else:
        added_keys.add(key)

len(weights)

Une fois le code ci-dessus exécuté, le dictionnaire des poids contient toutes les liaisons entre deux aéroports dont le poids est supérieur à 1 : seules les liaisons desservies par au moins 2 routes apparaissent.

Maintenant, traçons le graphique !

In [ ]:
import networkx as nx

# Importer networkx et initialiser le graphe
graph = nx.Graph()
# On garde une trace des nœuds ajoutés dans cet ensemble pour ne pas les ajouter 2 fois
nodes = set()

# Itérer sur chaque liaison
for k, weight in weights.items():
    try:
        # Séparer les ids de source et dest et les convertir en entier
        source, dest = k.split("_")
        source, dest = [int(source), int(dest)]

        # Ajouter source et dest s'ils ne sont pas déjà dans le graphe
        if source not in nodes:
            graph.add_node(source)
        if dest not in nodes:
            graph.add_node(dest)

        # Ajouter à la fois source et dest à l'ensemble nodes (un ensemble n'autorise pas les doublons)
        nodes.add(source)
        nodes.add(dest)

        # Ajouter la liaison au graphe
        graph.add_edge(source, dest, weight=weight)
    except (ValueError, IndexError):
        pass

# Positions des nœuds (la graine seed rend le dessin reproductible)
pos = nx.spring_layout(graph, seed=42)

# Dessiner les nœuds et les liaisons
plt.figure(figsize=(8, 6))
nx.draw_networkx_nodes(graph, pos, node_color="red", node_size=10, alpha=0.8)
nx.draw_networkx_edges(graph, pos, width=1.0, alpha=1)

# Afficher le graphique
plt.show()